# Tau Supersymmetry — Analysis Overview

<div style="text-align: justify">

This notebook serves as a guide to the <b>Tau Supersymmetry</b> search analysis. The analysis is implemented as a modular, multi-stage machine learning pipeline — from raw ROOT data through multiclass classification to statistical significance testing. Each stage is available both as an <b>interactive Jupyter notebook</b> (with inline plots and outputs) and as a <b>CLI script</b> for automated, reproducible execution.

</div>

## Pipeline Overview

The diagram below shows how the stages connect. Each notebook can be explored independently — outputs from previous stages are persisted as Parquet files.

```
ROOT ntuples
     │
     ▼
┌─────────────────────────────────┐
│  01  Preprocessing              │  ──▶  Selection cuts, event weights, ROOT → Parquet
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  02  Feature Engineering        │  ──▶  Physics features, class labels, sample merging
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  03  Exploratory Data Analysis  │  ──▶  Data quality, class balance, correlations, feature distributions
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  04  Hyperparameter Tuning      │  ──▶  Optuna search with k-fold CV and pruning
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  05a  BDT Training (XGBoost)    │  ──▶  Gradient-boosted decision trees
│  05b  DNN Training (PyTorch)    │  ──▶  Deep neural network classifier
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  06a  BDT Evaluation            │  ──▶  Feature importance, classification report, confusion matrix, ROC & PR curves, score distributions
│  06b  DNN Evaluation            │  ──▶  Permutation importance, classification report, confusion matrix, ROC & PR curves, score distributions
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  07  Regions                    │  ──▶  CR/VR/SR construction, CLs significance, kinematic distributions
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  Serving (FastAPI)              │  ──▶  REST API for real-time inference from trained models
└─────────────────────────────────┘
```

## Notebooks

| # | Notebook | Description | CLI Equivalent |
|---|----------|-------------|----------------|
| 01 | [Preprocessing](./01_preprocessing.ipynb) | Reads ROOT ntuples, applies selection cuts, computes event weights, and exports to Parquet | `python preprocess.py` |
| 02 | [Feature Engineering](./02_feature_engineering.ipynb) | Prepares physics-based input features, assigns class labels, and merges samples into a single DataFrame | `python feature_engineer.py` |
| 03 | [Exploratory Data Analysis](./03_eda.ipynb) | Examines data quality, class balance, feature correlations, and feature distributions | `python eda.py` |
| 04 | [Hyperparameter Tuning](./04_hyperparameter_tuning.ipynb) | Searches hyperparameter space using Optuna with TPE sampling, median pruning, and k-fold CV | `python tune.py` |
| 05a | [BDT Training (XGBoost)](./05a_bdt_training.ipynb) | Trains a gradient-boosted decision tree classifier with early stopping and MLflow tracking | `python train_bdt.py` |
| 05b | [DNN Training (PyTorch)](./05b_dnn_training.ipynb) | Trains a deep neural network classifier with mini-batch SGD, AMP, and early stopping | `python train_dnn.py` |
| 06a | [BDT Evaluation](./06a_bdt_evaluation.ipynb) | Evaluates the trained BDT with a comprehensive suite of metrics and visualisations | `python evaluate_bdt.py` |
| 06b | [DNN Evaluation](./06b_dnn_evaluation.ipynb) | Evaluates the trained DNN with a comprehensive suite of metrics and visualisations | `python evaluate_dnn.py` |
| 07 | [Regions](./07_regions.ipynb) | Constructs ML-based signal, control, and validation regions and computes CLs significance with pyhf | `python regions.py` |

## CLI Pipeline

The entire analysis can be run from the command line without opening any notebooks. Each script mirrors its notebook counterpart and is configured via [Hydra](https://hydra.cc/) YAML files in `configs/`.

```bash
# Run the full pipeline
make pipeline            # preprocess → feature-engineer → train

# Or run individual stages
make preprocess          # 01 — ROOT → Parquet
make feature-engineer    # 02 — feature extraction & sample merging
make eda                 # 03 — exploratory data analysis
make tune                # 04 — hyperparameter optimisation
make train               # 05a — BDT training
make regions             # 07 — ML-based regions & significance

# DVC-tracked reproducible run
make repro               # runs dvc repro
```

All hyperparameters can be overridden via the command line:

```bash
python train_bdt.py model.learning_rate=0.03 model.max_depth=5
python tune.py model=dnn tuning.n_trials=200 tuning.n_splits=3
```

## Model Serving

After training, models can be deployed as a REST API using FastAPI. The server loads a trained BDT or DNN checkpoint and serves predictions over HTTP.

```bash
# Start inference server for a trained BDT
python serve.py --model-type bdt --model-path models/bdt.ubj

# Start inference server for a trained DNN
python serve.py --model-type dnn --model-path models/dnn.pt --class-names background signal
```

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/health` | GET | Health check — is the server running? |
| `/model/info` | GET | Model metadata (expected features, class names) |
| `/predict` | POST | Classify a single event |
| `/predict/batch` | POST | Classify multiple events in one request |

```bash
# Example: classify a single event
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"features": {"met": 250.0, "jet_n": 3, "dphi": 1.2}}'

# Interactive API docs (auto-generated by FastAPI)
open http://localhost:8000/docs
```

The serving layer uses an **adapter pattern** (`src/serving/registry.py`) so that BDT and DNN models share a uniform interface — the API code never touches model-specific logic.

## Stack

| Category | Libraries |
|----------|----------|
| Data Processing | uproot, awkward-array, NumPy, SciPy, pandas, Numba, Pandera |
| Visualisation | Matplotlib, Seaborn, Plotly |
| Machine Learning | XGBoost, PyTorch, scikit-learn, Optuna, SHAP |
| Infrastructure | Docker, uv, GitHub Actions |
| Configuration & Reproducibility | Hydra, OmegaConf, DVC, MLflow |
| Serving / API | FastAPI, Uvicorn, Pydantic |
| Testing & QA | pytest, pytest-cov, mypy, Ruff, pre-commit |
| Physics | pyhf, atlas-mpl-style|